# Local `mlflow`

Goal:

- Retrain the YOLO model
- Track the training with mlflow


## Environment

Setup notebook and mlflow with `docker compose`

> Jupyter at http://127.0.0.1:8888. `MLFLOW_TRACKING_URI` is set by compose.


In [ ]:
from src.tracking import tracking_uri
import sys
from pathlib import Path

import mlflow
import torch
import ultralytics

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# print version
print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("ultralytics", ultralytics.__version__)
print("mlflow     ", mlflow.__version__)
print("cuda       ", torch.cuda.is_available())

# print mlflow uri
print("tracking   ", tracking_uri())

Define MLflow instance


In [ ]:
# define MLflow instance
mlflow.set_tracking_uri(tracking_uri())

# test by search experiments
print(mlflow.search_experiments())

## Data summary

Inspect the raw dataset before training.


In [ ]:
from src.data_loader import build_split, summarize, verify_split, write_data_yaml

# limit for fast smoke run; e.g. 200
# LIMIT = None  # unlimit
LIMIT = 200  # limit 200

# print summary
stats = summarize(RAW)
print({k: stats[k] for k in ("pairs", "boxes_total",
      "boxes_per_image_max", "malformed")})

# print split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=0))
print(verify_split(PROCESSED))

# print data config
names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

## Train with tracking

Use mlflow callback to pass training process


In [ ]:
import os
import time

import yaml
from ultralytics import YOLO

# experiment name
EXPERIMENT = "yolo-plate-detection"

# load parameters from config file
train_cfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text())
cfg = dict(train_cfg)
model_weights = cfg.pop("model")
cfg["project"] = str(ROOT / cfg["project"])

n_train = len(list((PROCESSED / "train" / "images").iterdir()))

# Set env var
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT
os.environ["MLFLOW_RUN"] = f"cpu-{n_train}img-{cfg['epochs']}ep-{cfg['imgsz']}px"

# Keep the run open
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "true"

print(f"experiment {EXPERIMENT}")
print(f"run        {os.environ['MLFLOW_RUN']}")

# construct model class with parameters
model = YOLO(model_weights)

# log start time
start = time.time()

# run the training process, get performance results
results = model.train(data=str(data_yaml), **cfg)

# print elapsed time
print(f"\nelapsed: {time.time() - start:.0f}s")

### MLflow log hyperparameters

Log parameters and terminate run


In [ ]:
from src.tracking import log_dataset_context

# Log hyperparameters to mlflow with helping function
logged = log_dataset_context(
    PROCESSED,
    RAW,
    **{"data.limit": str(LIMIT), "data.seed": 0, "run.device": "cpu"},
)

# get currently active MLflow run
run = mlflow.active_run()

# print id
print(f"run_id {run.info.run_id}")

# print parameters
for key, value in logged.items():
    print(f"  {key:26} {value}")

# terminates the currently active MLflow run 
mlflow.end_run()
print("\nrun closed")

Confirm parameters are logged.


In [ ]:
from src.tracking import latest_run_id

# get last run
run_id = latest_run_id(EXPERIMENT)
fetched = mlflow.get_run(run_id)

# print last run param
print(f"run_id  {run_id}")
print(f"status  {fetched.info.status}")

print("\nfinal metrics")
for key in sorted(fetched.data.metrics):
    if "mAP" in key or "precision" in key or "recall" in key:
        print(f"  {key:28} {fetched.data.metrics[key]:.4f}")

print("\ndataset params")
for key in sorted(k for k in fetched.data.params if k.startswith("data.")):
    print(f"  {key:28} {fetched.data.params[key]}")

print("\nartifacts")
for artifact in mlflow.artifacts.list_artifacts(run_id=run_id):
    print(f"  {artifact.path}")

Get historical metrics


In [ ]:
# Get historical metrics
client = mlflow.tracking.MlflowClient()  # clien interface
history = client.get_metric_history(run_id, "metrics/mAP50-95B")

print(f"{'epoch':>6} {'mAP50-95':>10}")
for point in history:
    print(f"{point.step:>6} {point.value:>10.4f}")

## List runs

Compare every run in the experiment, side by side.


In [ ]:
from src.tracking import compare_runs

compare_runs(EXPERIMENT)

Browse the same data at http://127.0.0.1:5000.